# Week4
## 1. Baseline Template -> Whitened Compressed Data Matching + Sorting

Goal: use templates from uncompressed baseline results as reference, run spike detection / template matching on whitened and compressed data, then continue sorting.

In [12]:
# Part A. Import whitened + compressed data first
import numpy as np
import pandas as pd
import torch
from pathlib import Path

ROOT = Path('F:/academic')
DATA_ROOT = ROOT / '.test_data'
BASE_BIN = DATA_ROOT / 'ZFM-02370_mini.imec0.ap.short.bin'

# Candidate compressed inputs: prefer reconstructed .bin for direct Kilosort usage
COMPRESSED_BIN_CANDIDATES = [
    ROOT / 'whitened_recon_ratio_0.10.bin',
    ROOT / 'whitened_recon_ratio_0.20.bin',
]

# Keep .npy assets for quick checks
WHITENED_NPY = ROOT / 'whitened_data.npy'
DCT_COEFF_010 = ROOT / 'whitened_dct_ratio_0.10_coeff.npy'
DCT_COEFF_020 = ROOT / 'whitened_dct_ratio_0.20_coeff.npy'

compressed_bin = next((f for f in COMPRESSED_BIN_CANDIDATES if f.exists()), None)
if compressed_bin is None:
    raise FileNotFoundError('Cannot find whitened_recon_ratio_*.bin. Generate reconstructed bin files in previous notebooks first.')

print('Compressed bin:', compressed_bin)
if WHITENED_NPY.exists():
    whitened_data = np.load(WHITENED_NPY, mmap_mode='r')
    print('whitened_data shape:', whitened_data.shape)
if DCT_COEFF_010.exists():
    print('found:', DCT_COEFF_010.name)
if DCT_COEFF_020.exists():
    print('found:', DCT_COEFF_020.name)



Compressed bin: F:\academic\whitened_recon_ratio_0.10.bin
whitened_data shape: (1350000, 385)
found: whitened_dct_ratio_0.10_coeff.npy
found: whitened_dct_ratio_0.20_coeff.npy


In [13]:
# Part B. Load baseline templates from uncompressed data
from kilosort.io import load_ops

results_dir = DATA_ROOT / 'kilosort4'
ops_base = load_ops(results_dir / 'ops.npy')
templates_base = np.load(results_dir / 'templates.npy')
st_base = np.load(results_dir / 'spike_times.npy').squeeze()
clu_base = np.load(results_dir / 'spike_clusters.npy').squeeze()

# Baseline universal templates / PCA if available in ops
wTEMP_base = ops_base.get('wTEMP', None)
wPCA_base = ops_base.get('wPCA', None)

print('baseline templates.npy shape:', templates_base.shape)
print('baseline spikes:', st_base.shape, 'clusters:', clu_base.shape)
print('ops has wTEMP:', wTEMP_base is not None, '| wPCA:', wPCA_base is not None)


baseline templates.npy shape: (267, 61, 383)
baseline spikes: (137381,) clusters: (137381,)
ops has wTEMP: True | wPCA: True


## 2. Run Spike Detect + Template Matching + Sorting on Compressed Data
Use the standard Kilosort pipeline here to keep behavior consistent with your other notebooks.


### Note: 383 vs 385 channels
Neuropixels raw workflows often keep extra non-neural channels (e.g., sync/reference/aux), so baseline pipelines may use 385 total channels.
The compressed reconstructed binaries here are 383-channel float32 data. This channel-count mismatch breaks direct use of the default 385-channel probe `chanMap`.

To keep the standard probe path compatible in this notebook, we pad the 383-channel compressed data to 385 channels by appending two zero channels as placeholders.
This does not add neural information; it only restores array shape compatibility for Kilosort I/O and probe indexing.


In [ ]:
from kilosort import run_kilosort

# Fixed from file inspection:
# whitened_recon_ratio_*.bin -> float32, 383 channels, 1,350,000 samples
src_n_chan = 383
target_n_chan = 385
compressed_dtype = np.float32

# Build a 385-channel padded binary once, so default Neuropixels probe chanMap stays valid.
padded_bin = ROOT / f'{compressed_bin.stem}.pad385.bin'
if not padded_bin.exists():
    src = np.memmap(compressed_bin, dtype=compressed_dtype, mode='r')
    n_samples = src.size // src_n_chan
    src2d = src[: n_samples * src_n_chan].reshape(n_samples, src_n_chan)

    dst = np.memmap(padded_bin, dtype=compressed_dtype, mode='w+', shape=(n_samples, target_n_chan))
    dst[:, :src_n_chan] = src2d
    dst[:, src_n_chan:] = 0.0
    dst.flush()
    del dst, src2d, src
    print('created padded bin:', padded_bin)
else:
    print('reuse padded bin:', padded_bin)

settings = {
    'filename': padded_bin,
    'n_chan_bin': target_n_chan,
    'fs': float(ops_base['fs']),
    # Reuse baseline-related parameters when possible
    'nt': int(ops_base.get('nt', 61)),
    'nt0min': int(ops_base.get('nt0min', 20)),
    'nearest_chans': int(ops_base.get('nearest_chans', 10)),
    'nearest_templates': int(ops_base.get('nearest_templates', 100)),
    'templates_from_data': True,
}

out_dir = ROOT / 'ks_compressed_baseline_init'
out_dir.mkdir(parents=True, exist_ok=True)

ops_cmp, st_cmp, clu_cmp, tF_cmp, Wall_cmp, similar_cmp, is_ref_cmp, est_contam_cmp, kept_cmp = run_kilosort(
    settings=settings,
    probe_name='NeuroPix1_default.mat',
    results_dir=out_dir,
    data_dtype=compressed_dtype,
    save_preprocessed_copy=False,
    do_CAR=False,
    invert_sign=False,
)

print('compressed sorting done')
print('st_cmp:', np.asarray(st_cmp).shape, 'clu_cmp:', np.asarray(clu_cmp).shape)


kilosort.run_kilosort: Kilosort version 4.1.7
kilosort.run_kilosort: Python version 3.11.15
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: System information:
kilosort.run_kilosort: Windows-10-10.0.26200-SP0 AMD64
kilosort.run_kilosort: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
kilosort.run_kilosort: Using GPU for PyTorch computations. Specify `device` to change this.
kilosort.run_kilosort: Using CUDA device: NVIDIA GeForce RTX 2050 4.00GB
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: Sorting [WindowsPath('F:/academic/whitened_recon_ratio_0.10.pad385.bin')]
kilosort.run_kilosort: Skipping common average reference.
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage before sorting
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    19.40 %
kilosort.run_kilosort: Mem used:     57.40 %     |       8.98 GB
kilosort.run_kiloso

created padded bin: F:\academic\whitened_recon_ratio_0.10.pad385.bin


kilosort.run_kilosort: Preprocessing filters computed in 3.00s; total 3.09s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    11.70 %
kilosort.run_kilosort: Mem used:     58.60 %     |       9.17 GB
kilosort.run_kilosort: Mem avail:     6.48 / 15.65 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   60.70 %     |      2.43   /     4.00 GB
kilosort.run_kilosort: Allocated:     0.39 %     |      0.02   /     4.00 GB
kilosort.run_kilosort: Max alloc:    30.63 %     |      1.22   /     4.00 GB
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort:  
kilosort.run_kilosort: Computing drift correction.
kilosort.run_kilosort: -----------------------------------

In [ ]:
# Part C. Baseline-anchored matching (test -> nearest baseline in time)
def _as_1d_times(st_like):
    st = np.asarray(st_like)
    if st.ndim == 1:
        return st.astype(np.int64, copy=False)
    return st[:, 0].astype(np.int64, copy=False)

def assign_to_baseline_clusters(baseline_st, baseline_clu, test_st, tolerance=30):
    baseline_st = np.asarray(baseline_st, dtype=np.int64)
    baseline_clu = np.asarray(baseline_clu, dtype=np.int64)
    test_st = np.asarray(test_st, dtype=np.int64)

    labels = np.full(test_st.shape[0], -1, dtype=np.int64)
    if baseline_st.size == 0 or test_st.size == 0:
        return labels

    pos = np.searchsorted(baseline_st, test_st)
    left = np.clip(pos - 1, 0, baseline_st.size - 1)
    right = np.clip(pos, 0, baseline_st.size - 1)

    dt_left = np.abs(test_st - baseline_st[left])
    dt_right = np.abs(test_st - baseline_st[right])
    use_left = dt_left <= dt_right
    nn = np.where(use_left, left, right)
    dt = np.abs(test_st - baseline_st[nn])

    ok = dt <= tolerance
    labels[ok] = baseline_clu[nn[ok]]
    return labels

baseline_st_1d = _as_1d_times(st_base)
test_st_1d = _as_1d_times(st_cmp)
test_clu_1d = np.asarray(clu_cmp).astype(np.int64)

labels_anchored = assign_to_baseline_clusters(baseline_st_1d, clu_base, test_st_1d, tolerance=30)
mask = labels_anchored >= 0

print('test spikes:', len(test_st_1d))
print('matched to baseline:', int(mask.sum()), f'({mask.mean()*100:.2f}%)')
print('baseline-anchored clusters:', len(np.unique(labels_anchored[mask])) if mask.any() else 0)


In [ ]:
# Part D. Save outputs for downstream evaluation
payload = {
    'baseline_templates_shape': templates_base.shape,
    'compressed_bin': str(compressed_bin),
    'st_cmp': np.asarray(st_cmp),
    'clu_cmp': np.asarray(clu_cmp),
    'labels_anchored_to_baseline': labels_anchored,
}

np.save(ROOT / 'week4_baseline_template_matching_sort.npy', payload, allow_pickle=True)
print('saved:', ROOT / 'week4_baseline_template_matching_sort.npy')
